In [1]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub robomimic torchvision wrapt pillow pandas diffusers
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')




fatal: destination path 'hnet' already exists and is not an empty directory.
fatal: destination path 'oat' already exists and is not an empty directory.
Cloning into 'congenial-adventure'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 238 (delta 94), reused 192 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (238/238), 241.66 KiB | 345.00 KiB/s, done.
Resolving deltas: 100% (94/94), done.
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Removed virtual environment at: .venv
Creating virtual environment at: .venv
Resolved 124 packages in 2.50s                                       ⠋ Resolving dependencies...                                                     
Prepared 121 packages in 38.31s                                          
░░░░░░░░░░░░░░░░░░░░ [0/121] Installing wheels...                               warning: Failed to hardlink files; falling back to f

In [2]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")


os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)



# Распаковываем архив прямо в ту же папку
!unzip -o -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

replace /kaggle/working/oat/data/libero/libero10_N500.zarr/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
drwxr-xr-x 4 root root       4096 Apr 12 21:50 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 12 21:50 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 12 21:49 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 12 21:58 /kaggle/working/oat/data/libero/README.md


In [2]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sebersehmer (sebersehmer-nopeinc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/oat/output/20260412/225721_train_oattok_libero10_N500/wandb/run-20260412_225733-ladyzfee
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 20260412.225721_train_oattok_libero10_N500
wandb: ⭐️ View project at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment
wandb: 🚀 View run at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment/runs/ladyzfee
wandb: ⢿ updating run metadata (0.0s)               
wandb: ⣻ updating run metadata (0.0s)
wandb: ⣽ updating run metadata (0.0s)
wandb: ⣾ updating run metadata (0.0s)
wandb: ⣷ uploading history steps 919-967, summary, console li

In [4]:
# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
!TOK=$(find /kaggle/working/outputs /kaggle/working/oat/outputs -name '*.ckpt' 2>/dev/null | sort | tail -1) && \
 MPLBACKEND=agg uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt=$TOK \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16

Seed set to 42
Error executing job with overrides: ['strategy=single_gpu', '+model.tokenizer_ckpt=/kaggle/working/oat/output/20260412/225721_train_oattok_libero10_N500/checkpoints/latest.ckpt', '+dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr', 'batch_size=16']
Traceback (most recent call last):
  File "/kaggle/working/run.py", line 30, in main
    system = LitSystem(cfg)
             ^^^^^^^^^^^^^^
  File "/kaggle/working/src/core/system.py", line 20, in __init__
    self.model = FDDRATPolicy(cfg.model, shape_meta=cfg.shape_meta)
                                                    ^^^^^^^^^^^^^^
omegaconf.errors.ConfigAttributeError: Key 'shape_meta' is not in struct
    full_key: shape_meta
    object_type=dict

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
